# Recuperação dos rejeitados por falso positivo (filtro anti-alucinação)

Pré-requisitos: sessão com projeto em `/content/AI-Orchestrator`, Ollama + microsserviços rodando (células 2–4 do notebook v3), Drive montado.

O que faz: aplica `build_dataset.py` corrigido → remove do `rejected.jsonl` as linhas com motivo "número ..." (libera os hashes) → re-roda pipeline (questions/routing pulam tudo; só ~320 trajetórias liberadas reprocessam, ~1h30) → assemble refaz train/val → backup no Drive.

**IMPORTANTE:** Todos os outputs intermediários e logs vão direto pro Drive. Se a sessão cair, o progresso persiste e o pipeline retoma de onde parou.

In [ ]:
# Célula 1 — sanidade: Drive montado, processo anterior morto, ambiente vivo
import os

DRIVE_DATASET = '/content/drive/MyDrive/ai-orchestrator-dataset'
DRIVE_LOG = f'{DRIVE_DATASET}/recover.log'

assert os.path.exists(DRIVE_DATASET), 'Monte o Drive primeiro'
assert os.path.exists('/content/AI-Orchestrator/train/build_dataset.py'), 'Projeto ausente — restaure a sessão'
!ps aux | grep build_dataset | grep -v grep || echo 'OK: nenhum build_dataset rodando'
print(f'Log será salvo em: {DRIVE_LOG}')

In [ ]:
# Célula 2 — aplica o build_dataset.py corrigido (vem do Drive)
!cp /content/drive/MyDrive/ai-orchestrator-dataset/build_dataset.py /content/AI-Orchestrator/train/build_dataset.py
!grep -n '_canonical_decimals' /content/AI-Orchestrator/train/build_dataset.py | head -3

In [ ]:
# Célula 3 — libera rejeitados com motivo "número ..." (falso positivo)
# Backup do rejected.jsonl no Drive ANTES de modificar
import json, pathlib, shutil

DRIVE_DATASET = '/content/drive/MyDrive/ai-orchestrator-dataset'
p = pathlib.Path('/content/AI-Orchestrator/train/dataset/rejected.jsonl')

# Backup local + Drive
shutil.copy(p, str(p) + '.bak')
shutil.copy(p, f'{DRIVE_DATASET}/rejected.jsonl.bak')
print(f'Backup salvo em {DRIVE_DATASET}/rejected.jsonl.bak')

rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
keep = [r for r in rows if not r.get('motivo', '').startswith('número')]
print(f'total={len(rows)} | liberados p/ reprocesso={len(rows)-len(keep)} | mantidos={len(keep)}')
p.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in keep))

In [ ]:
# Célula 4 — reprocessa só os liberados + re-assemble (background, ~1h30)
# Log vai direto pro Drive (persiste entre reconexões)
# Output do dataset também é copiado pro Drive a cada stage do pipeline
import subprocess, os

DRIVE_DATASET = '/content/drive/MyDrive/ai-orchestrator-dataset'
DRIVE_LOG = f'{DRIVE_DATASET}/recover.log'

# Wrapper: roda build_dataset + copia resultados pro Drive ao final
cmd = f"""cd /content/AI-Orchestrator && \
python3 train/build_dataset.py --stage all --target 3000 2>&1 | tee {DRIVE_LOG} && \
echo '=== BACKUP AUTOMÁTICO ===' >> {DRIVE_LOG} && \
cp train/dataset/*.jsonl {DRIVE_DATASET}/ 2>/dev/null && \
echo 'Backup automático concluído' >> {DRIVE_LOG} && \
ls -la {DRIVE_DATASET}/*.jsonl >> {DRIVE_LOG}"""

proc = subprocess.Popen(cmd, shell=True)
print(f'PID: {proc.pid}')
print(f'Log (persistente): {DRIVE_LOG}')
print('Acompanhe na célula 5. NÃO re-rode esta célula.')

In [ ]:
# Célula 5 — monitor (re-rode quando quiser; NÃO re-rode a célula 4)
# Lê log do Drive (funciona mesmo após reconexão de sessão)
DRIVE_DATASET = '/content/drive/MyDrive/ai-orchestrator-dataset'
DRIVE_LOG = f'{DRIVE_DATASET}/recover.log'

!tail -5 "$DRIVE_LOG"
!echo '---'
!wc -l /content/AI-Orchestrator/train/dataset/trajectories.jsonl /content/AI-Orchestrator/train/dataset/rejected.jsonl 2>/dev/null || echo 'Arquivos locais não encontrados (sessão resetou?) — cheque backup no Drive:'
!ls -la "$DRIVE_DATASET"/*.jsonl 2>/dev/null | tail -5

In [ ]:
# Célula 6 — verificação final + backup explícito (rode APÓS [assemble] aparecer no log)
import os

DRIVE_DATASET = '/content/drive/MyDrive/ai-orchestrator-dataset'
DRIVE_LOG = f'{DRIVE_DATASET}/recover.log'

# Checa se terminou
with open(DRIVE_LOG) as f:
    log = f.read()

if '[assemble] total=' in log or 'Backup automático concluído' in log:
    # Backup final explícito (redundante — célula 4 já faz, mas garante)
    !cp /content/AI-Orchestrator/train/dataset/*.jsonl "$DRIVE_DATASET/" 2>/dev/null
    print('BACKUP FINAL OK')
    !ls -la "$DRIVE_DATASET"/*.jsonl
else:
    print('AINDA NÃO TERMINOU — aguarde o [assemble] no monitor (célula 5)')
    !tail -3 "$DRIVE_LOG"